In [1]:
import os

In [2]:
%pwd

'c:\\Users\\LENOVO\\Fraud-Transaction-Detection\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\LENOVO\\Fraud-Transaction-Detection'

In [5]:
import pandas as pd
df = pd.read_pickle("processed/full_preprocessed_dataset.pkl")
print(df['TERMINAL_ID'].value_counts().head())
print(df['CUSTOMER_ID'].value_counts().head())


TERMINAL_ID
4018    376
692     372
5295    368
8130    360
872     356
Name: count, dtype: int64
CUSTOMER_ID
382     767
3864    762
2891    761
775     754
1411    752
Name: count, dtype: int64


In [6]:
high_fraud_tx = df[df['TX_FRAUD'] == 1].sample(1)
print(high_fraud_tx[['TERMINAL_ID', 'CUSTOMER_ID', 'TX_AMOUNT']])


       TERMINAL_ID CUSTOMER_ID  TX_AMOUNT
615186        6904         183       94.1


In [7]:
# Ensure datetime is parsed properly
df['TX_DATETIME'] = pd.to_datetime(df['TX_DATETIME'])

# Find Terminal IDs with at least 2 transactions
terminal_counts = df['TERMINAL_ID'].value_counts()
valid_terminals = terminal_counts[terminal_counts > 1].index.tolist()

# Find Customer IDs with at least 2 transactions
customer_counts = df['CUSTOMER_ID'].value_counts()
valid_customers = customer_counts[customer_counts > 1].index.tolist()

# Show a few valid Terminal IDs and Customer IDs
print("✅ Sample TERMINAL_IDs with history:")
print(valid_terminals[:5])

print("\n✅ Sample CUSTOMER_IDs with history:")
print(valid_customers[:5])

# (Optional) Show sample transactions from one of them
sample_terminal_id = valid_terminals[0]
sample_customer_id = valid_customers[0]

print(f"\n📍 Sample transactions for TERMINAL_ID = {sample_terminal_id}")
print(df[df['TERMINAL_ID'] == sample_terminal_id].sort_values("TX_DATETIME").head(5))

print(f"\n👤 Sample transactions for CUSTOMER_ID = {sample_customer_id}")
print(df[df['CUSTOMER_ID'] == sample_customer_id].sort_values("TX_DATETIME").head(5))


✅ Sample TERMINAL_IDs with history:
[4018, 692, 5295, 8130, 872]

✅ Sample CUSTOMER_IDs with history:
[382, 3864, 2891, 775, 1411]

📍 Sample transactions for TERMINAL_ID = 4018
       TRANSACTION_ID         TX_DATETIME CUSTOMER_ID TERMINAL_ID  TX_AMOUNT  \
3606             3606 2018-04-01 10:24:26        4716        4018       5.43   
6839             6839 2018-04-01 15:10:15        1621        4018      59.18   
9117             9117 2018-04-01 20:49:05        1423        4018      36.51   
13500           13500 2018-04-02 10:52:59        2403        4018      66.68   
13892           13892 2018-04-02 11:25:41        4141        4018      68.43   

      TX_TIME_SECONDS TX_TIME_DAYS  TX_FRAUD  TX_FRAUD_SCENARIO  TX_HOUR  \
3606            37466            0         0                  0       10   
6839            54615            0         0                  0       15   
9117            74945            0         0                  0       20   
13500          125579            1    

In [9]:
import pandas as pd

# Load your full preprocessed dataset
df = pd.read_pickle("processed/full_preprocessed_dataset.pkl")

# Ensure TX_DATETIME is in datetime format
df['TX_DATETIME'] = pd.to_datetime(df['TX_DATETIME'])

# Sort for consistent rolling behavior
df = df.sort_values(by='TX_DATETIME')

# Filter for fraudulent transactions only
fraud_df = df[df['TX_FRAUD'] == 1]

# Set the window period
window_days = 7

# Track results
results = []

# Loop through unique terminals that had frauds
for terminal_id in fraud_df['TERMINAL_ID'].unique():
    terminal_data = df[df['TERMINAL_ID'] == terminal_id].sort_values('TX_DATETIME')
    
    for i in range(len(terminal_data)):
        current_tx = terminal_data.iloc[i]
        current_time = current_tx['TX_DATETIME']
        
        # Get window of past 7 days (excluding current tx)
        past_window = terminal_data[
            (terminal_data['TX_DATETIME'] < current_time) &
            (terminal_data['TX_DATETIME'] >= current_time - pd.Timedelta(days=window_days))
        ]
        
        if past_window['TX_FRAUD'].sum() > 0:
            results.append({
                'TX_DATETIME': current_time,
                'TERMINAL_ID': current_tx['TERMINAL_ID'],
                'CUSTOMER_ID': current_tx['CUSTOMER_ID'],
                'TX_AMOUNT': current_tx['TX_AMOUNT'],
                'PAST_7D_FRAUDS': past_window['TX_FRAUD'].sum()
            })
            break  # One match is enough for each terminal

# Convert to DataFrame for display
results_df = pd.DataFrame(results)

# Display top 5 examples
print(results_df.head())


          TX_DATETIME  TERMINAL_ID  CUSTOMER_ID  TX_AMOUNT  PAST_7D_FRAUDS
0 2018-04-01 11:59:14         3059           55      36.28               1
1 2018-04-02 10:17:37         6050         2295      43.75               1
2 2018-04-03 09:39:52         9102         4070      39.64               1
3 2018-04-02 19:30:40         6893         4384       8.02               1
4 2018-04-03 11:32:58         1143         2446      18.19               1


In [10]:
import pandas as pd

# Load the full preprocessed dataset
df = pd.read_pickle("processed/full_preprocessed_dataset.pkl")

# Ensure datetime is in correct format
df['TX_DATETIME'] = pd.to_datetime(df['TX_DATETIME'])

# Filter for fraudulent transactions only
fraud_df = df[df['TX_FRAUD'] == 1]

# Group and list fraudulent transactions by terminal
fraud_by_terminal = fraud_df.groupby('TERMINAL_ID').apply(
    lambda x: x[['TX_DATETIME', 'CUSTOMER_ID', 'TX_AMOUNT']].sort_values('TX_DATETIME')
)

# Optional: reset index for cleaner view
fraud_by_terminal = fraud_by_terminal.reset_index(level=0)

# Show top few for verification
print(fraud_by_terminal.head(20))


         TERMINAL_ID         TX_DATETIME CUSTOMER_ID  TX_AMOUNT
561077             4 2018-05-29 11:18:47          79     288.15
265242             5 2018-04-28 13:57:29           1     205.35
338852             5 2018-05-06 09:20:16           1     273.60
1312272            6 2018-08-15 18:03:44         940     419.30
233887            13 2018-04-25 10:23:24        1273      63.70
98093             14 2018-04-11 08:04:38        4583     234.90
1200982           15 2018-08-04 08:39:28        4751      37.70
601309            19 2018-06-02 14:33:33        3296      81.50
1585875           22 2018-09-13 11:16:29        3570     291.20
1699439           23 2018-09-25 08:41:01        2271     129.45
1530329           24 2018-09-07 14:34:34        1864     144.15
1501631           26 2018-09-04 14:35:34        2684     140.65
126779            27 2018-04-14 07:43:34        2241     209.50
470461            29 2018-05-19 23:36:00        4974     280.36
529537            29 2018-05-26 07:09:44

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_11408\2237736559.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fraud_by_terminal = fraud_df.groupby('TERMINAL_ID').apply(
